# Rokoko Bake vs Reference -- MPJPE Evaluation

Compares joint world positions between a reference recording clip and a Rokoko-baked clip
of the same take, using **MPJPE** (Mean Per-Joint Position Error: the average Euclidean
distance, in meters, between corresponding joints of two poses, averaged over every joint
and every frame). Lower is closer to the reference.

Use this to check whether `RokokoJsonlBaker`'s temporal smoothing option
(`BakeOptions.temporalSmoothingRadius`, see
[`RokokoJsonlBaker.cs`](../Assets/Scripts/Rokoko/RokokoJsonlBaker.cs)) actually brings the
baked clip closer to the reference, or just blurs it.

## Producing the CSVs this notebook reads

The two clips being compared store completely different curve data --
`test001.anim` (the reference) has per-bone position/rotation Transform curves, while a
Rokoko bake has Humanoid muscle + root-motion curves -- so there's no way to compare them
by parsing the `.anim` YAML directly. Instead, use the
**`Rokoko > Export Clip Joint Positions to CSV...`** menu (added by
[`RokokoClipJointExporter.cs`](../Assets/Scripts/Rokoko/RokokoClipJointExporter.cs)) inside
Unity: it samples a clip on the actual character's Animator via `AnimationClip.SampleAnimation`
(the same evaluation Unity itself uses, so muscle curves go through the real Avatar
retargeting) and writes every Humanoid bone's world position to CSV.

Workflow:
1. Select the character GameObject the recording was made on (its Animator must be Humanoid,
   and its hierarchy must contain the `mixamorig1:...` bone names the reference clip's paths
   use -- `BlenderModel.prefab` looks like this character in this project).
2. Open `Rokoko > Export Clip Joint Positions to CSV...`, assign that GameObject and
   `Assets/Animations/RecordedAnimations/test001.anim`, keep the default 30 Hz sample rate,
   export to `documentation/mpjpe/test001_reference.csv`.
3. Bake `test001` with **Temporal Smoothing Radius = 0** (current default/off), export that
   clip to `documentation/mpjpe/test001_baked_raw.csv`.
4. Re-bake with a smoothing radius on (try 2-4), export that clip to
   `documentation/mpjpe/test001_baked_smoothed.csv`.
5. Run this notebook.

The config cell below already points at those three paths -- change them if you used
different names. The "raw" and "smoothed" comparisons run independently, so it's fine to run
this notebook after step 3 only, before step 4 exists.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CONFIG = {
    "reference_csv": "mpjpe/test001_reference.csv",
    "baked_raw_csv": "mpjpe/test001_baked_raw.csv",
    "baked_smoothed_csv": "mpjpe/test001_baked_smoothed.csv",
    # Must match (or be <=) the Sample Rate used when exporting, so the common time grid
    # below doesn't invent frames finer than what was actually exported.
    "sample_rate_hz": 30.0,
}


## Loading and resampling

Each CSV is long-format: one row per `(time, joint)` sample with that joint's world
`x, y, z` at that time. The reference and baked clips are two independent pipelines with
their own frame timings (see
[`RokokoJsonlBaker.cs`](../Assets/Scripts/Rokoko/RokokoJsonlBaker.cs)'s per-frame `dt`
derived from `studio_timestamp`), so rather than assume their sample times line up exactly,
each joint's track is linearly interpolated (`np.interp`) onto a shared time grid spanning
the overlap between both clips' durations.

In [ ]:
def load_joint_csv(path):
    df = pd.read_csv(path)
    df["joint"] = df["joint"].astype(str)
    return df


def resample_to_grid(df, joints, t_grid):
    """Returns {joint: (N, 3) array} linearly interpolated onto t_grid."""
    out = {}
    for joint in joints:
        sub = df[df["joint"] == joint].sort_values("time")
        if len(sub) < 2:
            continue
        t = sub["time"].to_numpy()
        out[joint] = np.stack([
            np.interp(t_grid, t, sub["x"].to_numpy()),
            np.interp(t_grid, t, sub["y"].to_numpy()),
            np.interp(t_grid, t, sub["z"].to_numpy()),
        ], axis=1)
    return out


def compute_mpjpe(reference_df, other_df, sample_rate_hz):
    """Returns (t_grid, per_frame_mpjpe, per_joint_mean_error, overall_mpjpe)."""
    joints = sorted(set(reference_df["joint"]) & set(other_df["joint"]))
    if not joints:
        raise ValueError("No joint names in common between the two CSVs.")

    t_max = min(reference_df["time"].max(), other_df["time"].max())
    t_grid = np.arange(0.0, t_max, 1.0 / sample_rate_hz)

    ref = resample_to_grid(reference_df, joints, t_grid)
    other = resample_to_grid(other_df, joints, t_grid)
    joints = sorted(set(ref) & set(other))

    # (frames, joints) matrix of per-joint Euclidean distance at each frame.
    errors = np.stack([np.linalg.norm(ref[j] - other[j], axis=1) for j in joints], axis=1)

    per_frame_mpjpe = errors.mean(axis=1)
    per_joint_mean_error = dict(zip(joints, errors.mean(axis=0)))
    overall_mpjpe = errors.mean()
    return t_grid, per_frame_mpjpe, per_joint_mean_error, overall_mpjpe


## Reference vs current bake (no smoothing)

In [ ]:
reference_df = load_joint_csv(CONFIG["reference_csv"])
baked_raw_df = load_joint_csv(CONFIG["baked_raw_csv"])

t_grid, per_frame_raw, per_joint_raw, mpjpe_raw = compute_mpjpe(
    reference_df, baked_raw_df, CONFIG["sample_rate_hz"])

print(f"Overall MPJPE (no smoothing): {mpjpe_raw * 100:.2f} cm  ({mpjpe_raw:.4f} m)")
print("\nWorst 5 joints:")
for joint, err in sorted(per_joint_raw.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {joint:<20} {err * 100:6.2f} cm")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t_grid, per_frame_raw * 100, label="baked (no smoothing)", color="#d95f02")
plt.xlabel("time (s)")
plt.ylabel("MPJPE (cm)")
plt.title("Per-frame MPJPE vs reference")
plt.legend()
plt.tight_layout()
plt.show()


## After temporal smoothing

Re-run the export for a clip baked with `Temporal Smoothing Radius` > 0 (step 4 above) to
`documentation/mpjpe/test001_baked_smoothed.csv`, then run the cells below. If that file
doesn't exist yet, this just prints a reminder instead of failing the rest of the
notebook.

In [ ]:
import os

smoothed_path = CONFIG["baked_smoothed_csv"]
if not os.path.exists(smoothed_path):
    print(f"No smoothed export found at '{smoothed_path}' yet -- "
          "bake with Temporal Smoothing Radius > 0, export it, and re-run this cell.")
else:
    baked_smoothed_df = load_joint_csv(smoothed_path)
    t_grid_s, per_frame_smoothed, per_joint_smoothed, mpjpe_smoothed = compute_mpjpe(
        reference_df, baked_smoothed_df, CONFIG["sample_rate_hz"])

    delta = mpjpe_smoothed - mpjpe_raw
    verdict = "CLOSER to reference" if delta < 0 else "FARTHER from reference"
    print(f"Overall MPJPE (no smoothing): {mpjpe_raw * 100:6.2f} cm")
    print(f"Overall MPJPE (smoothed):     {mpjpe_smoothed * 100:6.2f} cm")
    print(f"Delta: {delta * 100:+.2f} cm -- smoothing made the bake {verdict}")


In [ ]:
if os.path.exists(smoothed_path):
    plt.figure(figsize=(10, 4))
    plt.plot(t_grid, per_frame_raw * 100, label="baked (no smoothing)", color="#d95f02")
    plt.plot(t_grid_s, per_frame_smoothed * 100, label="baked (smoothed)", color="#1b9e77")
    plt.xlabel("time (s)")
    plt.ylabel("MPJPE (cm)")
    plt.title("Per-frame MPJPE vs reference -- before vs after smoothing")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if os.path.exists(smoothed_path):
    joints = sorted(set(per_joint_raw) & set(per_joint_smoothed),
                     key=lambda j: -per_joint_raw[j])[:10]
    x = np.arange(len(joints))
    width = 0.35

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, [per_joint_raw[j] * 100 for j in joints], width, label="no smoothing", color="#d95f02")
    plt.bar(x + width / 2, [per_joint_smoothed[j] * 100 for j in joints], width, label="smoothed", color="#1b9e77")
    plt.xticks(x, joints, rotation=45, ha="right")
    plt.ylabel("mean error (cm)")
    plt.title("Per-joint mean error -- worst 10 joints (by no-smoothing error)")
    plt.legend()
    plt.tight_layout()
    plt.show()
